# Cold-Start Forecasting & Clustering Pipeline
**Version:** `FINAL_P01`

## Overview
This notebook implements a clustering-based machine learning pipeline to forecast the total sales of new products during their launch phase (first 8 weeks). Because new products lack historical sales data (the "cold-start" problem), the pipeline relies heavily on launch-time features and historical behavior of similar products.

**Key Pipeline Characteristics:**
* **Aggregation Strategy:** Forecasts the *total aggregated sales* during the first 8 weeks rather than a week-by-week forecast.
* **Steady-State Exclusion:** Calculates time-series features using only the first `n` weeks of sales history. Long-term steady-state behavior is excluded to strictly model launch dynamics.
* **Granular Categorization:** Uses the full category path (`Category - sub_cat - sub_sub_cat`) to prevent misclassification of similar-sounding sub-categories.
* **Clustering Optimization:** Sweeps through $k$ values to find the optimal number of clusters. For similar WMAPE scores (within a 0.01 margin), the pipeline favors the lower $k$ for simplicity.
* **Output Generation:** Exports detailed product-level forecasts as well as macro-level metrics for performance analysis.

## Known Limitations & Future Work (To-Do)
* **Bias Analysis:** Further investigation into systemic forecast biases is required.
* **Unknown Features Handling:** Needs to be addressed, but it seems that within this pipeline, it is not feasible.
* **Similar Products Logic:** `num_similars` is currently computed based on predecessors. However, it does not perfectly account for whether those predecessors were still *active* at the time of the new product's launch.
* **Feature Refinements:** * Convert `num_similars` into categorical bins (Low, Medium, High).
    * Develop a more granular metric than `sub-sub-cat` to represent true product substitutes.
    * Address data leakage in `price_level` and utilize a more robust `price_index`.

---
## 1. Environment Setup & Imports

In [ ]:
# DATA HANDLING & MATH
import pandas as pd
import numpy as np

# PATH & TIME
from pathlib import Path
from datetime import datetime
import pytz
from google.colab import drive

# VISUALIZATION
import matplotlib.pyplot as plt
import seaborn as sns

# MACHINE LEARNING
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    adjusted_rand_score, accuracy_score, confusion_matrix,
    mean_squared_error, mean_absolute_error
)
from scipy.optimize import linear_sum_assignment

# PARALLEL EXECUTION
from joblib import Parallel, delayed
import multiprocessing

---
## 2. Directory & Path Configuration
Mounts Google Drive and establishes the necessary input/output directories for the datasets and forecast results.

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Define paths
folder_path = Path('/content/drive/MyDrive/folder_path')
out_folder  = Path('/content/drive/MyDrive/out_folder')

# Ensure directories exist
folder_path.mkdir(parents=True, exist_ok=True)
out_folder.mkdir(parents=True, exist_ok=True)

print(f'Input folder:  {folder_path}')
print(f'Output folder: {out_folder}')

---
## 3. Data Loading & Initial Preparation

This step handles the core data transformations. Note that `full_sub_cat` and `full_sub_sub_cat` are pure label concatenations without any sales aggregation, meaning they carry no sales signal and can be safely built on the full dataframe before splitting to prevent data leakage.

In [ ]:
def _prep_df(df, cols2drop, hz=8):
    """
    Prepares the main dataframe by filtering the forecast horizon, building
    categorical features, and calculating inflation-adjusted price indices.

    Args:
        df (pd.DataFrame): Raw transaction dataframe.
        cols2drop (list): List of columns to drop from the final dataframe.
        hz (int): The forecast horizon in weeks (default: 8).

    Returns:
        pd.DataFrame: Cleaned and feature-engineered dataframe.
    """
    # 1. Filter dataset to include only the first 'hz' weeks of sales per product
    df['weeks_since_first_sale'] = (df['invoice_date'] - df['first_sale']).dt.days // 7 + 1
    df = df[df['weeks_since_first_sale'] <= FORECAST_HORIZON_WEEKS].copy()
    df = df.drop(columns=['weeks_since_first_sale'])

    print(f'Filtered to {len(df):,} rows, representing the first {FORECAST_HORIZON_WEEKS} weeks of sales for each product.')

    df['item_quantity'] = df['item_quantity'].astype(int)

    # 2. Pure label concatenations (No aggregation/leakage risk)
    df['full_sub_cat']     = df['Category'] + ' - ' + df['sub cat']
    df['full_sub_sub_cat'] = df['full_sub_cat'] + ' - ' + df['sub sub cat']

    # 3. New Brand Feature: Identify if a product launch coincides with the brand's first-ever launch
    brand_launches = df.groupby('brand_name')['first_sale'].min().rename('brand_first_launch').reset_index()
    df = df.merge(brand_launches, on='brand_name', how='left')
    df['is_new_brand'] = (df['first_sale'] == df['brand_first_launch']).astype(int)
    df.drop(columns=['brand_first_launch'], inplace=True)

    # 4. Substitutability / Number of Similar Products
    product_launches = df[['product_code', 'full_sub_sub_cat', 'first_sale']].drop_duplicates()

    def count_predecessors(row, all_launches):
        """Counts how many products in the same sub-sub-category were launched prior."""
        return all_launches[
            (all_launches['full_sub_sub_cat'] == row['full_sub_sub_cat']) &
            (all_launches['first_sale'] < row['first_sale'])
        ]['product_code'].nunique()

    unique_products = product_launches.copy()
    unique_products['num_similars_new'] = unique_products.apply(
        count_predecessors, axis=1, all_launches=unique_products
    )

    df = df.merge(unique_products[['product_code', 'num_similars_new']], on='product_code', how='left')
    df['num_similars'] = df['num_similars_new']
    df.drop(columns=['num_similars_new'], inplace=True)

    # Bin substitutability
    def map_substitutability(n):
        if n <= 3: return 'subs_3'
        elif n <= 20: return 'subs_20'
        elif n <= 40: return 'subs_40'
        else: return 'subs_ultra'

    df['substitutability'] = df['num_similars'].apply(map_substitutability)

    # 5. Price Index Calculation (Inflation Adjusted)
    inflation_rates = {
        2021: 0.402, 2022: 0.458, 2023: 0.407,
        2024: 0.325, 2025: 0.509, 2026: 0.689
    }

    def get_cumulative_inflation_factor(target_year, target_month, base_year=2021):
        """Calculates the compounding inflation factor from the base year."""
        factor = 1.0
        for year in range(base_year, int(target_year)):
            factor *= (1 + inflation_rates.get(year, 0.0))

        annual_rate_target = inflation_rates.get(int(target_year), 0.0)
        monthly_rate_target = (1 + annual_rate_target)**(1/12) - 1
        factor *= (1 + monthly_rate_target) ** (int(target_month) - 1)
        return float(factor)

    # Remove unusual sales and zero-quantity transactions before calculating price index
    mask = (df['unusual_sales'].astype(str).str.lower() != 'yes') & (df['item_quantity'] > 0)
    df_filtered = df[mask].copy()

    if len(df_filtered) > 0:
        df_filtered['year'] = df_filtered['invoice_date'].dt.year
        df_filtered['month'] = df_filtered['invoice_date'].dt.month
        df_filtered['cum_inflation_factor'] = [
            get_cumulative_inflation_factor(y, m) for y, m in zip(df_filtered['year'], df_filtered['month'])
        ]

        # Adjust gross unit price by cumulative inflation
        df_filtered['adj_unit_gross'] = df_filtered['unit_gross'] / df_filtered['cum_inflation_factor']
        new_price_indices = df_filtered.groupby('product_code')['adj_unit_gross'].mean().rename('price_index')

        df = df.drop(columns=['price_index'], errors='ignore').merge(new_price_indices, on='product_code', how='left')
    else:
        print("Warning: Filtered dataset is empty. Check 'unusual_sales' and 'item_quantity' values.")

    # Drop redundant columns
    df_2 = df.drop(columns=cols2drop, errors='ignore')

    display(df_2.info())
    display(df_2.head())

    return df_2

---
## 4. Feature Engineering Functions
These utility functions extract statistical and behavioral features (e.g., daily activity, transaction medians, coefficient of variation) for individual products based on their transaction history.

In [ ]:
def _calculate_daily_product_activity(df_local):
    """Calculates daily quantity and holiday flags per (product, date)."""
    daily = df_local.groupby(['product_code', 'product_name', 'invoice_date']).agg(
        daily_quantity=('item_quantity', 'sum'),
        is_holiday=('holiday', 'max')
    ).reset_index()

    daily['is_zero_quantity_day'] = (daily['daily_quantity'] == 0).astype(int)
    daily['is_zero_quantity_working_day'] = ((daily['daily_quantity'] == 0) & (daily['is_holiday'] == 0)).astype(int)
    daily['is_transaction_day'] = (daily['daily_quantity'] > 0).astype(int)

    return daily


def _calculate_product_sale_dates(daily, products_base_subset):
    """Calculates product-level tenure, working days, zero-sales share, and launch month."""
    psd = daily.groupby(['product_code', 'product_name']).agg(
        min_date=('invoice_date', 'min'),
        max_date=('invoice_date', 'max'),
        total_holiday_days=('is_holiday', 'sum'),
        count_zero_quantity_days=('is_zero_quantity_day', 'sum'),
        zero_sales_working_days=('is_zero_quantity_working_day', 'sum')
    ).reset_index()

    psd['duration_days'] = (psd['max_date'] - psd['min_date']).dt.days + 1
    psd['total_working_days'] = np.maximum(psd['duration_days'] - psd['total_holiday_days'], 0)
    psd['share_zero_qty_working_days'] = (psd['zero_sales_working_days'] / psd['total_working_days']).fillna(0)
    psd['duration_weeks'] = np.minimum(np.maximum(psd['duration_days'] / 7.0, 1.0), float(FORECAST_HORIZON_WEEKS))

    # Incorporate launch month as a categorical feature
    psd = psd.merge(products_base_subset[['product_code', 'first_sale']], on='product_code', how='left')
    psd['launch_month'] = psd['first_sale'].dt.month.astype(str)
    psd = psd.drop(columns=['first_sale'])

    return psd


def _aggregate_core_features(df_local):
    """Aggregates total quantity, split-local price_index, and transaction count."""
    core = df_local.groupby('product_code').agg(
        total_sales_quantity=('item_quantity', 'sum'),
        price_index=('price_index', 'max')
    ).reset_index()

    txn = (df_local[df_local['item_quantity'] > 0]
           .groupby('product_code').size()
           .reset_index(name='num_sales_transactions'))

    return core.merge(txn, on='product_code', how='left')


def _calculate_discount_rate(df_local):
    """Calculates mean discount rate, restricted to rows with non-zero quantity."""
    return (df_local[df_local['item_quantity'] > 0]
            .groupby('product_code')['discount_rate']
            .mean()
            .reset_index(name='median_discount_rate'))


def _calculate_cv_quantity(df_local):
    """Calculates the coefficient of variation (CV) and median of weekly sales quantities."""
    weekly = (df_local.set_index('invoice_date')
              .groupby('product_code')['item_quantity']
              .resample('W').sum()
              .reset_index())

    cv = weekly.groupby('product_code')['item_quantity'].agg(['mean', 'std', 'median']).reset_index()
    cv.rename(columns={'median': 'median_weekly_quantity'}, inplace=True)
    cv['cv_quantity'] = (cv['std'] / cv['mean']).fillna(0)

    return cv[['product_code', 'cv_quantity', 'median_weekly_quantity']]


def _calculate_weekly_txn_median(df_local):
    """Calculates median weekly transaction count based on active days per week."""
    df_local = df_local.copy()
    df_local['is_active_day'] = (df_local['item_quantity'] > 0).astype(int)

    weekly = (df_local.set_index('invoice_date')
              .groupby('product_code')['is_active_day']
              .resample('W').sum()
              .reset_index(name='txn_count'))

    return (weekly.groupby('product_code')['txn_count']
            .median()
            .reset_index(name='median_weekly_transactions'))


def _merge_all_features(products_base, core, psd, discounts, cvs, weekly_txns):
    """Joins all feature tables together, dropping redundancies to prevent axis overlap."""
    out = core.copy()
    psd_subset = psd[['product_code', 'product_name', 'duration_days', 'duration_weeks',
                      'share_zero_qty_working_days', 'total_holiday_days', 'total_working_days', 'launch_month']]

    out = out.merge(psd_subset, on='product_code', how='left')
    out = out.merge(weekly_txns, on='product_code', how='left').fillna({'median_weekly_transactions': 0})
    out = out.merge(discounts, on='product_code', how='left').fillna({'median_discount_rate': 0})
    out = out.merge(cvs, on='product_code', how='left').fillna({'median_weekly_quantity': 0, 'cv_quantity': 0})

    overlap = [c for c in products_base.columns if c in out.columns and c != 'product_code']
    extra_drops = ['last_sale']
    cols_to_drop = list(set(overlap + extra_drops))

    base_cleaned = products_base.drop(columns=cols_to_drop, errors='ignore')
    return base_cleaned.merge(out, on='product_code', how='left')


def _generate_features_for_split(df_split, products_base):
    """Orchestrates all feature extraction helpers for a specific temporal data split."""
    daily = _calculate_daily_product_activity(df_split)
    psd   = _calculate_product_sale_dates(daily, products_base)
    core  = _aggregate_core_features(df_split)
    disc  = _calculate_discount_rate(df_split)
    cvs   = _calculate_cv_quantity(df_split)
    wtxn  = _calculate_weekly_txn_median(df_split)

    return _merge_all_features(products_base, core, psd, disc, cvs, wtxn)

---
## 5. Temporal Splitting
To ensure realistic cold-start evaluation, the data is split based on the chronologically ordered *first sale* timestamps of products, dividing the timeline strictly into Training, Validation, and Testing sets without data leakage.

In [ ]:
def _split_timestamps(df_full, BASE_COLS, test_share=0.2, val_share_of_train=0.2):
    """Calculates chronological cutoff dates (test_ts, validation_ts) based on product launch sequence."""
    products = df_full[BASE_COLS].drop_duplicates().sort_values('first_sale')
    n_total = len(products)

    if n_total == 0:
        return pd.Timestamp.max, pd.Timestamp.max

    # Calculate Test Set cutoff timestamp
    if test_share <= 0:
        test_ts = products['first_sale'].max() + pd.Timedelta(days=1)
    elif test_share >= 1:
        test_ts = products['first_sale'].min()
    else:
        test_idx = int(n_total * (1 - test_share))
        test_idx = max(0, min(test_idx, n_total - 1))
        test_ts = products.iloc[test_idx]['first_sale']

    # Calculate Validation Set cutoff from the remaining pool
    train_val = products[products['first_sale'] < test_ts]
    n_train_val = len(train_val)

    if n_train_val > 0:
        if val_share_of_train <= 0:
            validation_ts = train_val['first_sale'].max() + pd.Timedelta(days=1)
        elif val_share_of_train >= 1:
            validation_ts = train_val['first_sale'].min()
        else:
            val_idx = int(n_train_val * (1 - val_share_of_train))
            val_idx = max(0, min(val_idx, n_train_val - 1))
            validation_ts = train_val.iloc[val_idx]['first_sale']
    else:
        validation_ts = test_ts

    return test_ts, validation_ts


def get_df_splits_and_timestamps(df_full, BASE_COLS, test_share=0.2, val_share_of_train=0.2):
    """Splits the raw transaction DataFrame into Train, Validation, and Test."""
    test_ts, validation_ts = _split_timestamps(df_full, BASE_COLS, test_share, val_share_of_train)

    df_train = df_full[df_full['invoice_date'] < validation_ts].copy()
    df_val = df_full[(df_full['invoice_date'] >= validation_ts) & (df_full['invoice_date'] < test_ts)].copy()
    df_test = df_full[df_full['invoice_date'] >= test_ts].copy()

    return df_train, df_val, df_test, test_ts, validation_ts


def generate_product_features(df_full, BASE_COLS, test_share=0.2, val_share_of_train=0.2):
    """Generates the aggregated product-level feature tables for each split."""
    test_ts, validation_ts = _split_timestamps(df_full, BASE_COLS, test_share, val_share_of_train)

    COLS_TO_DROP = ['product_name', 'price_index', 'last_sale']
    products = df_full[BASE_COLS].drop_duplicates()
    products_clean = products.drop(columns=COLS_TO_DROP, errors='ignore')

    # Split the clean product pool
    p_train = products_clean[products_clean['first_sale'] < validation_ts].copy()
    p_val = products_clean[(products_clean['first_sale'] >= validation_ts) & (products_clean['first_sale'] < test_ts)].copy()
    p_test = products_clean[products_clean['first_sale'] >= test_ts].copy()

    # Split the raw data for aggregation processing
    df_train = df_full[df_full['invoice_date'] < validation_ts]
    df_val = df_full[(df_full['invoice_date'] >= validation_ts) & (df_full['invoice_date'] < test_ts)]
    df_test = df_full[df_full['invoice_date'] >= test_ts]

    return (
        _generate_features_for_split(df_train, p_train) if not p_train.empty else pd.DataFrame(),
        _generate_features_for_split(df_val, p_val) if not p_val.empty else pd.DataFrame(),
        _generate_features_for_split(df_test, p_test) if not p_test.empty else pd.DataFrame(),
    )

---
## 6. Clustering Utilities & Profiling

Columns that are high-cardinality identifiers, redundant, or post-launch behavioral signals are strictly dropped before clustering. `price_index` used here is the *split-local* calculation to prevent future leakage.

In [ ]:
def _prep_df_train_cluster(COLS_TO_DROP_FOR_CLUSTERING, products_with_features_train):
    """Prepares the training feature table exclusively for K-Means Clustering."""
    df_train_cluster = products_with_features_train.drop(
        columns=COLS_TO_DROP_FOR_CLUSTERING, errors='ignore'
    ).copy()

    print(f'Clustering feature table: {df_train_cluster.shape[0]:,} products, '
          f'{df_train_cluster.shape[1]} features (excluding product_code)')

    return df_train_cluster


def preprocess_features(df):
    """
    Standardizes numeric features and One-Hot Encodes categorical features.
    Excludes the 'product_code' identifier.
    """
    df_feats      = df.drop(columns=['product_code'])
    product_codes = df[['product_code']]

    cat_cols = df_feats.select_dtypes(include=['object', 'category']).columns
    num_cols = df_feats.select_dtypes(include=['int64', 'float64']).columns

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ],
        remainder='passthrough'
    )

    X = preprocessor.fit_transform(df_feats)
    feature_names = preprocessor.get_feature_names_out()

    df_preprocessed = pd.DataFrame(X, columns=feature_names, index=df_feats.index)
    return product_codes, df_preprocessed, preprocessor


def plot_elbow_curve(df_preprocessed, k_range=(1, 12)):
    """Plots Inertia vs. k to visually aid in the selection of the optimal cluster count."""
    K_values = range(k_range[0], k_range[1] + 1)
    inertias = [
        KMeans(n_clusters=k, random_state=42, n_init=10).fit(df_preprocessed).inertia_
        for k in K_values
    ]

    plt.figure(figsize=(10, 5))
    plt.plot(K_values, inertias, marker='o')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Inertia')
    plt.title('Elbow Curve — K-Means Clustering')
    plt.xticks(K_values)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def perform_kmeans_clustering(df_train_cluster, k=3, plot_elbow=False):
    """Executes the K-Means clustering pipeline and assigns labels to the training data."""
    product_codes, df_preprocessed, preprocessor = preprocess_features(df_train_cluster)

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    df_clustered = df_train_cluster.copy()
    df_clustered['cluster'] = kmeans.fit_predict(df_preprocessed)

    if plot_elbow:
        plot_elbow_curve(df_preprocessed)

    return df_clustered, preprocessor, kmeans


# Optional: Toggle flag_run to True to visually profile specific cluster results post-sweep
flag_run = False
if flag_run:
    num_cols_cl = df_clustered.select_dtypes(include=['float64', 'int64']).columns.drop('cluster', errors='ignore')
    print(f"Cluster sizes (k={df_clustered['cluster'].nunique()}):")
    display(df_clustered['cluster'].value_counts().sort_index())

    # Add other profiling plots here...

---
## 7. Forecasting & Evaluation Utilities
* **WMAPE:** Weighted Mean Absolute Percentage Error.
* **Cluster-Mean Baseline Forecast:** Computes the historical mean of the **8-week total** quantity across all training products belonging to a cluster. New products classified into that cluster receive this mean as their static 8-week forecast.

In [ ]:
def wmape(actual, predicted):
    """Calculates Weighted Mean Absolute Percentage Error. Returns NaN on division by zero."""
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    total = actual.sum()
    return np.nan if total == 0 else np.abs(actual - predicted).sum() / total


def find_best_label_mapping(true_labels, pred_labels):
    """
    Maximizes agreement between classifier predictions and direct K-means labeling
    using the Hungarian assignment algorithm to align label spaces.
    """
    unique_true = np.unique(true_labels)
    unique_pred = np.unique(pred_labels)

    t_map = {v: i for i, v in enumerate(unique_true)}
    p_map = {v: i for i, v in enumerate(unique_pred)}

    mapped_true = np.array([t_map[l] for l in true_labels])
    mapped_pred = np.array([p_map[p] for p in pred_labels])

    cm = confusion_matrix(mapped_true, mapped_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)

    best_map = {unique_pred[col_ind[i]]: unique_true[row_ind[i]] for i in range(len(row_ind))}
    return np.array([best_map.get(p, -1) for p in pred_labels])


def calculate_wmape_baseline(df_train_full, df_validation_full, df_clustered_k, new_products_val_k):
    """
    Evaluates the static 8-week Cluster-Mean baseline against new product reality.
    """
    # Step A: 8-week aggregate for training items
    train_weekly = (df_train_full.set_index('invoice_date')
                    .groupby('product_code')['item_quantity']
                    .resample('W').sum().reset_index())

    train_weekly['week_num'] = train_weekly.groupby('product_code').cumcount() + 1
    train_8w = (train_weekly[train_weekly['week_num'] <= 8]
                .groupby('product_code')['item_quantity']
                .sum().reset_index(name='total_8w_qty'))

    train_8w = train_8w.merge(df_clustered_k[['product_code', 'cluster']], on='product_code', how='inner')

    # Step B: Calculate the mean cluster expectation
    cluster_baseline = (train_8w.groupby('cluster')['total_8w_qty']
                        .mean().round(0).astype(int)
                        .rename('baseline_total_8w_qty').reset_index())

    # Step C: Extract reality (Actual 8-week sales) for new products
    new_val_codes = new_products_val_k['product_code'].unique()
    val_weekly = (df_validation_full[df_validation_full['product_code'].isin(new_val_codes)]
                  .set_index('invoice_date')
                  .groupby('product_code')['item_quantity']
                  .resample('W').sum().reset_index())

    val_weekly['week_num'] = val_weekly.groupby('product_code').cumcount() + 1
    val_8w = (val_weekly[val_weekly['week_num'] <= 8]
              .groupby('product_code')['item_quantity']
              .sum().reset_index(name='actual_total_8w_qty'))

    # Step D: Compare Expectations vs Reality
    val_8w = val_8w.merge(
        new_products_val_k[['product_code', 'predicted_cluster_classifier']].rename(
            columns={'predicted_cluster_classifier': 'cluster'}
        ), on='product_code', how='left')

    val_8w = val_8w.merge(cluster_baseline, on='cluster', how='left')
    val_8w['baseline_total_8w_qty'] = val_8w['baseline_total_8w_qty'].fillna(0).astype(int)

    overall_wmape = wmape(val_8w['actual_total_8w_qty'], val_8w['baseline_total_8w_qty'])
    return overall_wmape, val_8w

---
## 8. K-Sweep Execution Loop
For each proposed number of clusters ($k$):
1. **Cluster:** Fits K-Means strictly on training sales.
2. **Train Classifier:** Trains a Logistic Regression model to predict the K-Means cluster using *only* launch-time features (`Region`, `price_index`, `full_sub_cat`).
3. **Classify:** Predicts the cluster for *new* validation products.
4. **Forecast & Evaluate:** Applies the historical mean of the predicted cluster as the forecast and computes accuracy metrics (WMAPE, RMSE, MAE, Bias).

In [ ]:
def _prep_feed_loop(products_with_features_val, products_with_features_train, df_train_cluster, EXCLUDE_FROM_CLASSIFIER):
    """Isolates strictly new validation products and defines allowable classification features."""
    new_products_val_df = products_with_features_val[
        ~products_with_features_val['product_code'].isin(products_with_features_train['product_code'])
    ].copy()

    print(f'New products isolated in validation set: {len(new_products_val_df):,}')

    clustering_features = [col for col in df_train_cluster.columns if col != 'product_code']
    CLASSIFIER_FEATURES = [f for f in clustering_features if f not in EXCLUDE_FROM_CLASSIFIER]

    return new_products_val_df, CLASSIFIER_FEATURES


def run_single_k(k_val, df_train_cluster_in, new_products_val_df, df_train_full_in, df_validation_full_in, CLASSIFIER_FEATURES):
    """Executes the pipeline for a single 'k' configuration."""
    # 1. K-Means
    df_clustered, _, kmeans_model = perform_kmeans_clustering(df_train_cluster_in, k=k_val, plot_elbow=False)

    # 2. Logistic Regression (Launch-Feature Classifier)
    X_train_cls = df_clustered[CLASSIFIER_FEATURES].copy()
    y_train_cls = df_clustered['cluster']

    num_cls_features = X_train_cls.select_dtypes(include=np.number).columns.tolist()
    cat_cls_features = X_train_cls.select_dtypes(exclude=np.number).columns.tolist()

    cls_pipeline = Pipeline([
        ('preprocessor', ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), num_cls_features),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cls_features)
            ], remainder='passthrough'
        )),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    cls_pipeline.fit(X_train_cls, y_train_cls)

    # 3. Classify New Products
    X_val_cls = new_products_val_df[CLASSIFIER_FEATURES].copy()
    new_products_val_k = new_products_val_df.copy()
    new_products_val_k['predicted_cluster_classifier'] = cls_pipeline.predict(X_val_cls)

    # 4. Evaluate Forecast
    overall_wmape, detailed_val_results = calculate_wmape_baseline(
        df_train_full_in, df_validation_full_in, df_clustered, new_products_val_k
    )

    act = detailed_val_results['actual_total_8w_qty']
    pred = detailed_val_results['baseline_total_8w_qty']

    return {
        'k': k_val,
        'wmape': overall_wmape,
        'rmse': np.sqrt(mean_squared_error(act, pred)),
        'mae': mean_absolute_error(act, pred),
        'bias': (pred.sum() - act.sum()) / act.sum() if act.sum() != 0 else np.nan,
        'detailed_results': detailed_val_results
    }


def run_loop(k_min, k_max, df_train_cluster, new_products_val_df, df_train_full_in, df_validation_full_in, CLASSIFIER_FEATURES):
    """Distributes the K-sweep across available CPU cores."""
    num_cores = multiprocessing.cpu_count()
    parallel_results = Parallel(n_jobs=num_cores, verbose=10)(
        delayed(run_single_k)(k, df_train_cluster, new_products_val_df, df_train_full_in, df_validation_full_in, CLASSIFIER_FEATURES)
        for k in range(k_min, k_max)
    )

    # Process Parallel Results
    all_detailed_val_results = {res['k']: res['detailed_results'] for res in parallel_results}
    results = [{k: v for k, v in res.items() if k != 'detailed_results'} for res in parallel_results]

    results_df = pd.DataFrame(results)
    sorted_results = results_df.sort_values(by=['wmape', 'k'])

    # Extract Optimal 'k' (favoring lower k for similar WMAPEs)
    sort_round_results = sorted_results.copy().round(2).sort_values(by=['wmape', 'k'])
    opt_k = int(sort_round_results.iloc[0]['k'])
    opt_k_detailed_val_results = all_detailed_val_results[opt_k]

    print(f"\nSelection complete. Optimal k: {opt_k}")
    display(sorted_results[['k', 'wmape', 'rmse', 'mae', 'bias']].head(10))

    return all_detailed_val_results, sorted_results, opt_k, opt_k_detailed_val_results

---
## 9. Execution Wrapper
Ties the entire pipeline together: loads, processes, splits, models, evaluates, and conditionally exports the results to Google Drive.

In [ ]:
def wrapper(
    df, k_min, k_max, FORECAST_HORIZON_WEEKS,
    COLS2DROP_BASE, BASE_COLS, COLS_TO_DROP_FOR_CLUSTERING, EXCLUDE_FROM_CLASSIFIER,
    test_share=0.2, val_share_of_train=0.2, flag_export=False
):
    """Main execution function mapping all sub-components of the pipeline."""
    # 1. Base Preprocessing
    df = _prep_df(df, COLS2DROP_BASE, FORECAST_HORIZON_WEEKS)

    # 2. Chronological Splits & Feature generation
    df_train_full, df_validation_full, df_test_full, test_ts, validation_ts = get_df_splits_and_timestamps(
        df, BASE_COLS, test_share=test_share, val_share_of_train=val_share_of_train
    )
    products_with_features_train, products_with_features_val, products_with_features_test = generate_product_features(
        df, BASE_COLS, test_share=test_share, val_share_of_train=val_share_of_train
    )

    print(f'\nSplits Information:')
    print(f'Train products      : {products_with_features_train.shape[0]:,}')
    print(f'Validation products : {products_with_features_val.shape[0]:,}')
    print(f'Test products       : {products_with_features_test.shape[0]:,}')
    print(f'Validation cutoff   : {validation_ts.date()}')
    print(f'Test cutoff         : {test_ts.date()}\n')

    # 3. Handle Cluster Data Preparation
    df_train_cluster = _prep_df_train_cluster(COLS_TO_DROP_FOR_CLUSTERING, products_with_features_train)

    # Correlation check/dropping (Handle 100% correlated columns manually detected)
    df_train_cluster.drop(columns=['median_weekly_transactions', 'share_zero_qty_working_days'], errors='ignore', inplace=True)

    # 4. Global K-Sweep Analysis
    new_products_val_df, CLASSIFIER_FEATURES = _prep_feed_loop(
        products_with_features_val, products_with_features_train, df_train_cluster, EXCLUDE_FROM_CLASSIFIER
    )

    all_detailed, sorted_results, opt_k, opt_k_detailed = run_loop(
        k_min, k_max, df_train_cluster, new_products_val_df, df_train_full, df_validation_full, CLASSIFIER_FEATURES
    )

    # 5. Export Results
    if flag_export:
        france_timezone = pytz.timezone('Europe/Paris')
        current_time = datetime.now(france_timezone).strftime('%Y%m%d_%H%M')

        macro_path = out_folder / f'clustering_forecasting_results_{current_time}.xlsx'
        sorted_results.to_excel(macro_path, index=False)
        print(f'Macro results exported to: {macro_path}')

        detailed_path = out_folder / f'clustering_forecasting_results_detailed_k{opt_k}_{current_time}.xlsx'
        opt_k_detailed_val_exp = opt_k_detailed[['product_code', 'cluster', 'baseline_total_8w_qty', 'actual_total_8w_qty']].copy()
        opt_k_detailed_val_exp.to_excel(detailed_path, index=False)
        print(f'Detailed results (k={opt_k}) exported to: {detailed_path}')

    return all_detailed, sorted_results, opt_k, opt_k_detailed

In [ ]:
def _features_show(df_train_cluster, CLASSIFIER_FEATURES):
    """It shows the features present in "Clustering" and "Classification"."""
    clustering_features = [col for col in df_train_cluster.columns if col != 'product_code']
    classifier_features = CLASSIFIER_FEATURES

    print(f"--- Features used for Clustering (k-means) [{len(clustering_features)}] ---")
    print(clustering_features)

    print(f"\n--- Features used for Classification (Logistic Regression) [{len(classifier_features)}] ---")
    print(classifier_features)

    # Identifying the overlap/differences
    only_clustering = set(clustering_features) - set(classifier_features)
    only_classifier = set(classifier_features) - set(clustering_features)

    if only_clustering:
        print(f"\nFeatures in Clustering but NOT in Classifier:\n{list(only_clustering)}")
    if only_classifier:
        print(f"\nFeatures in Classifier but NOT in Clustering:\n{list(only_classifier)}")

---
## 10. Execution (Validation)
Finds the `opt_k` using validation set.

In [ ]:
# ---------------------------------------------------------
# GLOBAL PARAMETERS & EXECUTION
# ---------------------------------------------------------
k_min = 2
k_max = 100

test_share = 0.2
val_share_of_train = 0.2
FORECAST_HORIZON_WEEKS = 8

COLS2DROP_BASE = [
    'is_corrected_cq', 'is_corrected_cq_opposite', 'price_level',
    'is_corrected_only_price', 'last_sale', 'sub cat', 'sub sub cat',
]

BASE_COLS = [
    'product_code', 'product_name', 'brand_name', 'Producer', 'Region',
    'price_index', 'Category', 'full_sub_sub_cat',
    'num_similars', 'substitutability', 'first_sale', 'full_sub_cat', 'is_new_brand'
]

COLS_TO_DROP_FOR_CLUSTERING = [
    'product_name', 'sub cat', 'sub sub cat', 'first_sale',
]

# Specifically drop behavioral/future features from classification at launch
EXCLUDE_FROM_CLASSIFIER = [
    'num_sales_transactions', 'duration_days', 'duration_weeks',
    'share_zero_qty_working_days', 'total_holiday_days', 'total_working_days', 'total_sales_quantity',
    'median_weekly_transactions', 'median_discount_rate', 'cv_quantity', 'median_weekly_quantity'
]

# LOAD MASTER DATA
df = pd.read_pickle(folder_path / 'final_merged_dataset.pkl')
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns.')

# RUN PIPELINE
all_detailed_val_results, sorted_results, opt_k, opt_k_detailed_val_results = wrapper(
    df, k_min, k_max,
    FORECAST_HORIZON_WEEKS,
    COLS2DROP_BASE,
    BASE_COLS,
    COLS_TO_DROP_FOR_CLUSTERING,
    EXCLUDE_FROM_CLASSIFIER,
    test_share=test_share,
    val_share_of_train=val_share_of_train,
    flag_export=False
)

---
## 11. Execution (Test)
Re-trains the pipeline using the `opt_k` using "train + validation" set. Then, applies it on the "test" set.

In [ ]:
all_detailed_val_results, sorted_results, opt_k, opt_k_detailed_val_results = wrapper(
    df, opt_k, (opt_k+1),
    FORECAST_HORIZON_WEEKS,
    COLS2DROP_BASE,
    BASE_COLS,
    COLS_TO_DROP_FOR_CLUSTERING,
    EXCLUDE_FROM_CLASSIFIER,
    test_share=0,
    val_share_of_train=0.2,
    flag_export=False
    )

**VAL RESULTS:**
* opt_k = 27
* Products Count: 785 (train: 3133)
* Date: 2023-10-22 --- 2024-12-10
* WMAPE: 0.744
* RMSE: 10.6
* MAE: 3.5
* Bias: -0.35

**TEST RESULTS:**
* k = 27
* Products Count: 995 (train: 3918)
* Date: 2024-12-10 --- 2026-04-29
* WMAPE: 0.934
* RMSE: 61.8
* MAE: 5.1
* Bias: -0.42

Questions and Doubts:
1. should we choose fewer clusters for a similar accuracy?
2. should we include price_index AND price_level? num_similars AND substitutability? etc.
3. should we keep correlated columns? total_quantity AND median_weekly_quantity?
4. should we keep all hierarchy of categories? cat, sub-cat, sub-sub-cat, etc.?
5. New value for categorical.
6. correlated features.